In [2]:
!pip install wandb

In [3]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f74b8a14d7be1ea92958af60b0b08fb0bb072c47c0c030da14a7046648eeaae7
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


In [4]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration, AdamW, get_linear_schedule_with_warmup
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
import wandb
from rouge_score import rouge_scorer
# from dotenv import load_dotenv
import os
from tabulate import tabulate
from nltk.translate.bleu_score import corpus_bleu
import sympy as sp
# load_dotenv()

In [12]:
df = pd.read_csv('/kaggle/input/rag-cleaned-second/cleaned_RAG_data.csv')

# If 'Unnamed: 0' column still exists, you can drop it
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# 80% -> Training Data, 20% -> Testing Data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 90% -> Training Data, 10% -> Validation Data
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

In [13]:
test_df.to_csv("test_df.csv")

In [14]:
df.head()

,Question,Object,Answer,Context
0,what is a yali ?,yali,a yali is a mythical creature found predominan...,yali is a mythical creature found predominantl...
1,what animals is a yali typically composed of ?,yali,a yali is typically depicted as a composite of...,yali is a mythical creature found predominantl...
2,what attributes does a yali symbolize ?,yali,"a yali symbolizes attributes like strength, pr...",yali is a mythical creature found predominantl...
3,why are yalis considered unique ?,yali,yalis are unique because they do not adhere to...,yali is a mythical creature found predominantl...
4,which cultures have similar mythical creatures...,yali,cultures that have similar mythical creatures ...,it shares similarities with other mythical cre...


In [15]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

def calculate_max_length(column_name):
    df[column_name] = df[column_name].astype(str)
    return df[column_name].apply(lambda x: len(tokenizer.tokenize(x))).max()

max_length_question = calculate_max_length('Question')
max_length_object = calculate_max_length('Object')
max_length_answer = calculate_max_length('Answer')
max_length_context = calculate_max_length('Context')

print(f"Maximum token length in the 'Question' column: {max_length_question + max_length_object + max_length_context}")
print(f"Maximum token length in the 'Answer' column: {max_length_answer}")

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Maximum token length in the 'Question' column: 189
Maximum token length in the 'Answer' column: 62


In [16]:
model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

In [17]:
class Dataset(Dataset):
    '''For Loading the datasetS! '''
    def __init__(self, data, question_max_length=200, answer_max_length=80):
        self.data = data
        self.tokenizer = tokenizer
        self.question_max_length = question_max_length
        self.answer_max_length = answer_max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        question = self.data.iloc[idx, 0]
        object_name = self.data.iloc[idx, 1]
        answer = self.data.iloc[idx, 2]
        context = self.data.iloc[idx, 3]
        combined = str(object_name) + ' ' + str(question) + ''  + str(context)

        inputs = self.tokenizer(
            combined,
            max_length=self.question_max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        targets = self.tokenizer(
            answer,
            max_length=self.answer_max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        input_ids = inputs.input_ids.squeeze()
        attention_mask = inputs.attention_mask.squeeze()
        target_ids = targets.input_ids.squeeze()

        return {

            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': target_ids
        }

In [18]:
model

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_lay

In [19]:
train_dataset = Dataset(train_df)
val_dataset = Dataset(val_df)
test_dataset = Dataset(test_df)

In [20]:
from rouge_score import rouge_scorer
# from nltk.translate.meteor_score import meteor_score
from sklearn.metrics import accuracy_score
import numpy as np
import math
import torch
from tqdm import tqdm
import pandas as pd
from tabulate import tabulate
import wandb
from nltk.translate.bleu_score import corpus_bleu

class VQA_Trainer:
    '''Class for Trainer Setup to Train the BART Model for VQA'''

    def __init__(self, model, train_dataloader, eval_dataloader, device, config):
        ''' Constructor '''
        self.model = model
        self.train_dataloader = train_dataloader
        self.eval_dataloader = eval_dataloader
        self.device = device
        self.tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')
        self.scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)  # Initialize ROUGE scorer

        # self.optimizer = AdamW(self.model.parameters())
        self.optimizer = AdamW(self.model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
        self.scheduler = get_linear_schedule_with_warmup(self.optimizer, num_warmup_steps=config['warmup_steps'], num_training_steps=len(self.train_dataloader) * config['epochs'])

        self.model.config.dropout = config['dropout']
        self.model.config.attention_dropout = config['attention_dropout']
        self.num_beams = config['num_beams']

        wandb.init(project=config['project_name'], config=config)
        wandb.watch(self.model, log="all")

    def evaluate(self):
        ''' For Evaluation at the End of Each Epoch '''
        self.model.eval()
        total_loss = 0
        predictions = []
        references = []
        token_level_accuracies = []
        exact_match_accuracies = []

        progress_bar = tqdm(self.eval_dataloader, desc="Evaluating")

        for batch in progress_bar:
            with torch.no_grad():
                inputs = {key: val.to(self.device) for key, val in batch.items()}
                outputs = self.model(**inputs)
                total_loss += outputs.loss.item()

                summary_ids = self.model.generate(inputs['input_ids'], max_length=80, num_beams=self.num_beams, early_stopping=True)
                decoded_preds = self.tokenizer.batch_decode(summary_ids, skip_special_tokens=True)

                labels = batch['labels']
                labels = torch.where(labels != -100, labels, self.tokenizer.pad_token_id)
                decoded_refs = self.tokenizer.batch_decode(labels, skip_special_tokens=True)

                predictions.extend([pred.split() for pred in decoded_preds])
                references.extend([[ref.split()] for ref in decoded_refs])

                for pred, ref in zip(decoded_preds, decoded_refs):
                    pred_tokens = pred.split()
                    ref_tokens = ref.split()
                    
                    # Token-level accuracy
                    token_accuracy = sum(1 for p, r in zip(pred_tokens, ref_tokens) if p == r) / max(len(ref_tokens), 1)
                    token_level_accuracies.append(token_accuracy)
                    
                    # Exact Match Accuracy
                    exact_match_accuracy = 1 if pred.strip() == ref.strip() else 0
                    exact_match_accuracies.append(exact_match_accuracy)

        avg_loss = total_loss / len(self.eval_dataloader)
        perplexity = math.exp(avg_loss)

        bleu_score = corpus_bleu(references, predictions)

        rouge_1_f1_scores = []
        rouge_2_f1_scores = []
        rouge_l_f1_scores = []
        for pred, ref in zip(decoded_preds, decoded_refs):
            rouge_1 = self.scorer.score(ref, pred)['rouge1']
            rouge_2 = self.scorer.score(ref, pred)['rouge2']
            rouge_l = self.scorer.score(ref, pred)['rougeL']
            
            rouge_1_f1_scores.append(rouge_1.fmeasure)
            rouge_2_f1_scores.append(rouge_2.fmeasure)
            rouge_l_f1_scores.append(rouge_l.fmeasure)

        avg_token_level_accuracy = np.mean(token_level_accuracies)
        avg_exact_match_accuracy = np.mean(exact_match_accuracies)

        avg_rouge_1_f1 = np.mean(rouge_1_f1_scores)
        avg_rouge_2_f1 = np.mean(rouge_2_f1_scores)
        avg_rouge_l_f1 = np.mean(rouge_l_f1_scores)

        # METEOR Score Calculation
        # meteor_scores = [meteor_score([ref], pred) for ref, pred in zip(decoded_refs, decoded_preds)]
        # avg_meteor = np.mean(meteor_scores)

        df = pd.DataFrame({
            "Training Loss": [avg_loss],
            "ROUGE-1 (F1)": [avg_rouge_1_f1],
            "ROUGE-2 (F1)": [avg_rouge_2_f1],
            "ROUGE-L (F1)": [avg_rouge_l_f1],
            "Corpus BLEU": [bleu_score],
            "Token-Level Accuracy": [avg_token_level_accuracy],
            "Exact Match Accuracy": [avg_exact_match_accuracy],
            # "METEOR": [avg_meteor],
            "Perplexity": [perplexity]
        })

        print(tabulate(df, headers="keys", tablefmt="psql"))

        metrics = {
            "Validation Loss": avg_loss,
            "perplexity": perplexity,
            "bleu": bleu_score,
            "rouge_1_f1": avg_rouge_1_f1,
            "rouge_2_f1": avg_rouge_2_f1,
            "rouge_l_f1": avg_rouge_l_f1,
            "token_level_accuracy": avg_token_level_accuracy,
            "exact_match_accuracy": avg_exact_match_accuracy,
            # "meteor": avg_meteor,
        }

        wandb.log(metrics)

        return metrics

    def train_epoch(self):
        ''' To Train for Single Epoch '''
        self.model.train()
        for batch in tqdm(self.train_dataloader, desc="Training"):
            self.optimizer.zero_grad()
            inputs = {key: val.to(self.device) for key, val in batch.items()}
            outputs = self.model(**inputs)
            loss = outputs.loss
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            wandb.log({"train_loss": loss.item()})

    def train(self, epochs):
        ''' To Train for N Number of Epochs Passed from User '''
        for epoch in range(epochs):
            print(f'Epoch {epoch+1}/{epochs}')
            self.train_epoch()
            metrics = self.evaluate()
            print(f"Metrics: {metrics}")
            torch.save(self.model.state_dict(), f"model_{epoch}.pth")


In [21]:
api_key = os.getenv('API_KEY')
!wandb login --relogin $api_key

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
Traceback (most recent call last):
  File "/opt/conda/bin/wandb", line 8, in <module>
    sys.exit(cli())
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1157, in __call__
    return self.main(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1078, in main
    rv = self.invoke(ctx)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1688, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1434, in invoke
    return ctx.invoke(self.callback, **ctx.params)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 783, in invoke
    return __callback(*args, **kwargs)
  File "/opt/conda/lib/python3.10/sit

In [22]:
config = {
     "epochs" : 10,
     "model_name": "facebook/bart-base",
     "project_name": "VQA_BART_RAG_second_DS",
    'learning_rate': 9.071880672175604e-05, 
    'weight_decay': 1.0818912251075672e-05, 
    'dropout': 0.4153380025576099, 
    'attention_dropout': 0.22747699513739159, 
    'num_beams': 4, 
    'batch_size': 8, 
    'warmup_steps': 124
}

In [23]:
train_dataloader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=True)

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BartForConditionalGeneration.from_pretrained(config['model_name']).to(device)

In [25]:
trainer = VQA_Trainer(model, train_dataloader, val_dataloader, device=device, config=config)

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wand

  ········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [ ]:
trainer.train(config['epochs'])

Epoch 1/10


Evaluating: 100%|██████████| 119/119 [01:27<00:00,  1.35it/s]


+----+-----------------+----------------+----------------+----------------+---------------+------------------------+------------------------+--------------+
|    |   Training Loss |   ROUGE-1 (F1) |   ROUGE-2 (F1) |   ROUGE-L (F1) |   Corpus BLEU |   Token-Level Accuracy |   Exact Match Accuracy |   Perplexity |
|----+-----------------+----------------+----------------+----------------+---------------+------------------------+------------------------+--------------|
|  0 |        0.468721 |        0.40111 |       0.175926 |       0.364073 |       0.18691 |               0.160873 |              0.0105597 |      1.59795 |
+----+-----------------+----------------+----------------+----------------+---------------+------------------------+------------------------+--------------+
Metrics: {'Validation Loss': 0.46872142383030485, 'perplexity': 1.5979497860139367, 'bleu': 0.1869098368299405, 'rouge_1_f1': 0.4011096116359274, 'rouge_2_f1': 0.17592592592592593, 'rouge_l_f1': 0.36407257459889036,

Evaluating: 100%|██████████| 119/119 [01:28<00:00,  1.34it/s]


+----+-----------------+----------------+----------------+----------------+---------------+------------------------+------------------------+--------------+
|    |   Training Loss |   ROUGE-1 (F1) |   ROUGE-2 (F1) |   ROUGE-L (F1) |   Corpus BLEU |   Token-Level Accuracy |   Exact Match Accuracy |   Perplexity |
|----+-----------------+----------------+----------------+----------------+---------------+------------------------+------------------------+--------------|
|  0 |        0.377675 |       0.554037 |       0.359307 |       0.554037 |      0.247163 |               0.206279 |              0.0158395 |      1.45889 |
+----+-----------------+----------------+----------------+----------------+---------------+------------------------+------------------------+--------------+
Metrics: {'Validation Loss': 0.37767515613251373, 'perplexity': 1.4588889547636006, 'bleu': 0.24716316787357492, 'rouge_1_f1': 0.5540372670807453, 'rouge_2_f1': 0.35930735930735924, 'rouge_l_f1': 0.5540372670807453,

Training:  51%|█████     | 542/1065 [01:46<01:42,  5.08it/s]

In [ ]:
wandb.finish()